# Facebook Live Sellers (Thailand) — Engagement Analysis

This notebook analyzes the **Facebook Live Sellers in Thailand** dataset (UCI ML Repository): 7,050 posts from 10 Thai fashion/cosmetics retailers, covering reactions, comments, shares, and the reaction-type breakdown (likes, loves, wows, hahas, sads, angrys).

It answers the questions from the dataset brief:
1. Does time of upload affect `num_reactions`?
2. How do `num_reactions`, `num_comments`, and `num_shares` correlate?
3. Train a K-Means clustering model on the engagement features.
4. Use the elbow method to choose the number of clusters.
5. What's the count of each post type?
6. What's the average `num_reactions`, `num_comments`, `num_shares` per post type?


## 1. Setup and data loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

pd.set_option('display.max_columns', None)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

In [ ]:
# If running in Colab, upload Facebook_Marketplace_data.csv first:
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv("Facebook_Marketplace_data.csv")
df.shape

## 2. Data cleaning

The raw file has 16 columns; `Column1`–`Column4` are fully empty (a known artifact of this dataset export) and are dropped, leaving the 14 meaningful attributes described in the data dictionary.

In [ ]:
print(df.isnull().sum())
print()
print("Column1-4 non-null counts:", df[['Column1','Column2','Column3','Column4']].count().tolist())

In [ ]:
df = df.drop(columns=['Column1', 'Column2', 'Column3', 'Column4'])

# Parse the publish timestamp
df['status_published'] = pd.to_datetime(df['status_published'], format='%m/%d/%Y %H:%M')
df['hour'] = df['status_published'].dt.hour
df['day_of_week'] = df['status_published'].dt.day_name()
df['year'] = df['status_published'].dt.year

print("Duplicate status_id rows:", df['status_id'].duplicated().sum())
print("Date range:", df['status_published'].min(), "to", df['status_published'].max())
df.head()

## 3. Post type counts (Question 5)

In [ ]:
post_type_counts = df['status_type'].value_counts()
post_type_counts

In [ ]:
ax = post_type_counts.reindex(['photo','video','status','link']).plot(
    kind='bar', color=['#2F3C7E','#F96167','#F9E795','#888780'], width=0.6
)
ax.set_ylabel("Number of posts")
ax.set_title("Post count by status type")
for i, v in enumerate(post_type_counts.reindex(['photo','video','status','link'])):
    ax.text(i, v + 60, f"{v:,}", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Average engagement by post type (Question 6)

In [ ]:
avg_by_type = df.groupby('status_type')[['num_reactions','num_comments','num_shares']].mean().round(1)
avg_by_type.reindex(['photo','video','status','link'])

In [ ]:
avg_plot = avg_by_type.reindex(['photo','video','status','link'])
avg_plot.plot(kind='bar', figsize=(8,5), color=['#2F3C7E','#F96167','#F9E795'])
plt.ylabel("Average count")
plt.title("Average reactions, comments, and shares by post type")
plt.xticks(rotation=0)
plt.legend(['Reactions','Comments','Shares'], frameon=False)
plt.tight_layout()
plt.show()

**Observation:** Status updates draw the highest average reactions, but videos dominate comments and shares by a wide margin — video content invites discussion and sharing in a way photo/status posts don't.

## 5. Does time of upload affect num_reactions? (Question 1)

In [ ]:
hourly = df.groupby('hour')['num_reactions'].mean()

colors = ['#F96167' if h in [17,18,19,20] else '#2F3C7E' for h in hourly.index]
plt.figure(figsize=(9,5))
plt.bar(hourly.index, hourly.values, color=colors, width=0.7)
plt.xlabel("Hour of day (24h)")
plt.ylabel("Average reactions")
plt.title("Average reactions by hour of publication")
plt.xticks(range(0,24,2))
plt.tight_layout()
plt.show()

In [ ]:
daily = df.groupby('day_of_week')['num_reactions'].mean().reindex(
    ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
)
daily.round(1)

**Observation:** Reactions trend higher for posts published in the evening (roughly 5–8pm), consistent with people browsing social media after work/school. Thursday is the strongest day on average and Tuesday the weakest, though the effect size is modest relative to the dataset's overall variance.

## 6. Correlation between reactions, comments, and shares (Question 2)

In [ ]:
engagement_cols = ['num_reactions','num_comments','num_shares','num_likes','num_loves','num_wows','num_hahas','num_sads','num_angrys']
corr = df[engagement_cols].corr()
corr.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(engagement_cols)))
ax.set_yticks(range(len(engagement_cols)))
labels = [c.replace('num_','') for c in engagement_cols]
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
for i in range(len(engagement_cols)):
    for j in range(len(engagement_cols)):
        ax.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center',
                color='white' if abs(corr.iloc[i,j])>0.5 else 'black', fontsize=8)
plt.colorbar(im, fraction=0.046, pad=0.04)
plt.title("Correlation matrix: engagement metrics")
plt.tight_layout()
plt.show()

In [ ]:
print(f"reactions vs comments: r = {corr.loc['num_reactions','num_comments']:.3f}")
print(f"reactions vs shares:   r = {corr.loc['num_reactions','num_shares']:.3f}")
print(f"comments vs shares:    r = {corr.loc['num_comments','num_shares']:.3f}")
print(f"reactions vs likes:    r = {corr.loc['num_reactions','num_likes']:.3f}")

**Observation:**
- `num_reactions` and `num_likes` are nearly identical (r ≈ 0.99) — likes are the overwhelming majority of all reactions.
- `num_comments` and `num_shares` are moderately correlated (r ≈ 0.64) — posts that get discussed also tend to get shared.
- `num_reactions` correlates only weakly with `num_comments` (r ≈ 0.15) and `num_shares` (r ≈ 0.25) — a post can rack up reactions without driving conversation or shares, and vice versa. These represent largely independent dimensions of engagement.

## 7. K-Means clustering (Question 3)

In [ ]:
feature_cols = ['num_reactions','num_comments','num_shares','num_likes',
                'num_loves','num_wows','num_hahas','num_sads','num_angrys']

X = df[feature_cols].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled.shape

### Elbow method (Question 4)

In [ ]:
inertias = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(K_range, inertias, marker='o', color='#2F3C7E', linewidth=2.5,
         markersize=7, markerfacecolor='#F96167', markeredgecolor='#F96167')
plt.axvline(x=4, color='#F9E795', linestyle='--', linewidth=2)
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia (within-cluster sum of squares)")
plt.title("Elbow method for optimal k")
plt.xticks(K_range)
plt.tight_layout()
plt.show()

for k, i in zip(K_range, inertias):
    print(f"k={k}: inertia={i:,.1f}")

**Observation:** Inertia drops sharply from k=1 to k=3, then the rate of decrease flattens noticeably. **k=4** is a reasonable elbow point — enough clusters to separate distinct engagement patterns without over-fragmenting the data.

### Fit final model (k=4) and profile the clusters

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

df['cluster'].value_counts().sort_index()

In [ ]:
cluster_profile = df.groupby('cluster')[feature_cols].mean().round(1)
cluster_profile

In [ ]:
pd.crosstab(df['cluster'], df['status_type'])

In [ ]:
cluster_colors = {0: '#888780', 1: '#F96167', 2: '#2F3C7E', 3: '#F9E795'}
cluster_labels = {0: 'Typical posts', 1: 'Viral discussions',
                   2: 'High-reaction, low-talk', 3: 'Super-engaged outliers'}

plt.figure(figsize=(9,6))
for c in sorted(df['cluster'].unique()):
    sub = df[df['cluster'] == c]
    plt.scatter(sub['num_reactions']+1, sub['num_comments']+1, s=18, alpha=0.6,
                color=cluster_colors[c], label=f"{cluster_labels[c]} (n={len(sub)})")
plt.xscale('log')
plt.yscale('log')
plt.xlabel("Reactions (log scale)")
plt.ylabel("Comments (log scale)")
plt.title("K-Means clusters: reactions vs. comments")
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

**Cluster interpretation:**

| Cluster | Size | Profile |
|---|---|---|
| 0 — Typical posts | 6,322 (90%) | Low-to-moderate engagement across the board; the baseline post |
| 1 — Viral discussions | 330 | Moderate reactions but very high comments (~2,951 avg) — almost entirely video posts that spark conversation |
| 2 — High-reaction, low-talk | 372 | High reactions (~1,823 avg) but few comments — posts people like/react to but don't discuss |
| 3 — Super-engaged outliers | 26 | Small group with extreme reactions AND comments — the rare breakout viral post |

This segmentation is useful for a seller: cluster 1/3 content (mostly video) drives conversation and shareability, while cluster 2 content drives passive approval (reactions) without engagement depth.

## 8. Summary of findings

1. **Timing matters moderately**: evening posts (5–8pm) and Thursdays see the highest average reactions, but the effect is secondary to content type.
2. **Reactions ≠ conversation**: `num_reactions` correlates weakly with `num_comments`/`num_shares` (r ≈ 0.15–0.25); `num_likes` drives almost all of `num_reactions` (r ≈ 0.99).
3. **Video content drives comments and shares**: videos average 642 comments and 116 shares per post, dwarfing other post types — despite photos being the most common format (61% of posts).
4. **Four natural clusters emerge** from K-Means: a large baseline group, a "viral discussion" group (mostly video), a "high-reaction, low-talk" group, and a tiny super-engaged outlier group.
5. **Status type counts**: 4,288 photos, 2,334 videos, 365 status updates, 63 links.
